# Exercise 06 Notebook: Validation, Testing, and Troubleshooting

- Why: Convert test workflow into repeatable, cell-wise checks.
- Outcome: You can isolate failures and prove rerun success.

In [ ]:
from pathlib import Path
import subprocess

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'tests').exists():
            return candidate
    raise RuntimeError('Could not locate repo root')

REPO_ROOT = find_repo_root(Path.cwd())

def run_cmd(cmd: list[str], cwd: Path = REPO_ROOT, allow_fail: bool = False):
    print('>', ' '.join(cmd))
    proc = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode != 0 and not allow_fail:
        raise RuntimeError(f'Command failed with exit code {proc.returncode}')
    return proc.returncode

## Step 1: Verify test assets
Why: Confirms available test scope before running pytest.
Outcome: You know which tests exist in this repo snapshot.

In [ ]:
print('Repo root:', REPO_ROOT)
for p in sorted((REPO_ROOT / 'tests').glob('test_*.py')):
    print('-', p.name)

## Step 2: Run non-live tests
Why: Establish local quality baseline independent of cloud dependencies.
Outcome: You get reliable pass/fail local signal.

In [ ]:
run_cmd(['uv', 'sync', '--extra', 'dev'])
rc = run_cmd(['uv', 'run', 'pytest', 'tests', '-m', 'not live'], allow_fail=True)
print('Exit code:', rc)
if rc != 0:
    print('Falling back to full tests because marker selection may be unavailable')
    run_cmd(['uv', 'run', 'pytest', 'tests'])

## Step 3: Run live path if configured
Why: Validates cloud-backed readiness.
Outcome: You confirm live path success or capture exact blocker.

In [ ]:
rc_live = run_cmd(['uv', 'run', 'pytest', 'tests', '-m', 'live'], allow_fail=True)
print('Live marker exit code:', rc_live)
if rc_live != 0:
    print('Running fallback live demo path')
    run_cmd(['uv', 'run', 'python', 'demo/04_cosmosdb.py'], allow_fail=True)

## Step 4: Troubleshooting pattern
Why: Guided labs need a repeatable diagnose-fix-rerun process.
Outcome: You can document one complete incident for Exercise 06 deliverable.

If a command fails:
1. Capture full error output
2. Apply one targeted fix
3. Re-run original command
4. Record before/after evidence